# ⚓ BITÁCORA DE RESPUESTAS
Construcción paso a paso de métricas.

In [1]:
import sys
import os
import pandas as pd

# 1. Configuración de Rutas
if os.path.abspath("..") not in sys.path:
    sys.path.insert(0, os.path.abspath(".."))

from database.db import engine

print("Entorno listo.")

# 2. Carga Base (Ajustada para Empresa)
query = """
SELECT o.*, e.nombre as empresa
FROM ofertas o
LEFT JOIN empresas e ON o.empresa_id = e.id
"""

try:
    df = pd.read_sql(query, engine)
    print(f"Carga Base Exitosa: {len(df)} ofertas registradas.")
except Exception as e:
    print(f"Error en la conexión: {e}")

Entorno listo.
Carga Base Exitosa: 58 ofertas registradas.


## 1. Origen del Proceso

Identifica el flujo o proceso de recolección de la oferta laboral.

<div style="font-size: 0.8em; line-height: 1.8; color: #ffffffff; font-family: monospace; text-align: right; padding-right: 10px; margin-top: 10px;">
df: Objeto DataFrame (tabla completa).<br>
column: Argumento que recibe el nombre de la columna (string).<br>
df[column]: Selección que genera una Series (una sola columna).<br>
.value_counts(): Método que cuenta repeticiones de valores únicos.<br>
</div>

In [2]:
# 1.1 Frecuencia
def calculate_frequency(df, column):
    return df[column].value_counts()

print("Distribución por origen de proceso (Conteo):")
print(calculate_frequency(df, 'origen_proceso'))

Distribución por origen de proceso (Conteo):
origen_proceso
fullstack    20
dds          20
dds_full     18
Name: count, dtype: int64


<div style="font-size: 0.8em; line-height: 1.8; color: #ffffffff; font-family: monospace; text-align: right; padding-right: 10px; margin-top: 10px;">
normalize=True: Convierte el conteo en valores relativos (0 a 1).<br>
* 100: Transforma la proporción en porcentaje.
</div>

In [3]:
# 1.2 Porcentaje
def calculate_distribution(df, column):
    return df[column].value_counts(normalize=True) * 100

print("Distribución por origen de proceso (Porcentaje %):")
print(calculate_distribution(df, 'origen_proceso'))

Distribución por origen de proceso (Porcentaje %):
origen_proceso
fullstack    34.482759
dds          34.482759
dds_full     31.034483
Name: proportion, dtype: float64


<div style="font-size: 0.8em; line-height: 1.8; color: #ffffffff; font-family: monospace; text-align: right; padding-right: 10px; margin-top: 10px;">
.mode(): Retorna los valores más frecuentes (Series).<br>
[0]: Accede al primer elemento de la moda detectada.
</div>

In [4]:
# 1.3 Moda
def calculate_mode(df, column):
    return df[column].mode()[0]

print(f"El origen de proceso con mayor volumen es: {calculate_mode(df, 'origen_proceso')}")

El origen de proceso con mayor volumen es: dds


<div style="font-size: 0.8em; line-height: 1.8; color: #ffffffff; font-family: monospace; text-align: right; padding-right: 10px; margin-top: 10px;">
.isna(): Genera una máscara booleana de valores nulos.<br>
.mean(): En booleanos, calcula la proporción de aciertos (True).
</div>

In [5]:
# 1.4 Calidad (Null Ratio)
def calculate_null_ratio(df, column):
    return df[column].isna().mean() * 100

print(f"Porcentaje de valores nulos en origen_proceso: {calculate_null_ratio(df, 'origen_proceso'):.2f}%")

Porcentaje de valores nulos en origen_proceso: 0.00%


## 2. Empresa

Nombre de la organización o agencia que publica la vacante.

<div style="font-size: 0.8em; line-height: 1.8; color: #ffffffff; font-family: monospace; text-align: right; padding-right: 10px; margin-top: 10px;">
.head(n): Retorna los primeros n elementos de la colección.
</div>

In [6]:
# 2.1 Frecuencia (Top 10)
def get_top_n(df, column, n=10):
    return df[column].value_counts().head(n)

print("Top 10 empresas con más ofertas:")
print(get_top_n(df, 'empresa'))

Top 10 empresas con más ofertas:
empresa
ASIGNAR                     2
Alianza Temporal            2
Gi Group Colombia           2
GENTE UTIL                  2
Belltech Colombia           2
Latin Promo Multimedia      1
Staffing de Colombia        1
Lexco                       1
POSITIVO S+ IT SOLUTIONS    1
AUDISOFT CONSULTING         1
Name: count, dtype: int64


<div style="font-size: 0.8em; line-height: 1.8; color: #ffffffff; font-family: monospace; text-align: right; padding-right: 10px; margin-top: 10px;">
Integridad: Reutilización de lógica de nulos aplicada a 'empresa'.
</div>

In [7]:
# 2.2 Integridad
print(f"Porcentaje de ofertas sin empresa especificada: {calculate_null_ratio(df, 'empresa'):.2f}%")

Porcentaje de ofertas sin empresa especificada: 15.52%


<div style="font-size: 0.8em; line-height: 1.8; color: #ffffffff; font-family: monospace; text-align: right; padding-right: 10px; margin-top: 10px;">
.groupby(): Agrupa el DataFrame por categorías.<br>
.all(): Valida si todos los registros del grupo cumplen la condición.
</div>

In [8]:
# 2.3 Segmentación (Exclusividad Inglés)
def get_exclusive_companies(df, company_col, condition_col):
    res = df.groupby(company_col)[condition_col].all()
    return res[res].index.tolist()

exclusive_list = get_exclusive_companies(df, 'empresa', 'requiere_ingles')
print(f"Cantidad de empresas con inglés exclusivo: {len(exclusive_list)}")
print("Lista de empresas:")
print(exclusive_list)

Cantidad de empresas con inglés exclusivo: 4
Lista de empresas:
['CONATEMPO S. A. S', 'Grupo Vicca INVERSIONES SUPER ROYAL CARIBE S A', 'Latin Promo Multimedia', 'TURISVIVIENDA']


<div style="font-size: 0.8em; line-height: 1.8; color: #ffffffff; font-family: monospace; text-align: right; padding-right: 10px; margin-top: 10px;">
.nunique(): Cuenta el número de elementos únicos en la serie.
</div>

In [9]:
# 2.4 Unicidad
def get_unique_count(df, column):
    return df[column].nunique()

print(f"Total de empresas únicas detectadas: {get_unique_count(df, 'empresa')}")

Total de empresas únicas detectadas: 44


<div style="font-size: 0.8em; line-height: 1.8; color: #ffffffff; font-family: monospace; text-align: right; padding-right: 10px; margin-top: 10px;">
(counts == 1).sum(): Filtra y cuenta elementos con una sola ocurrencia.
</div>

In [10]:
# 2.5 Análisis de Larga Cola
def analyze_long_tail(df, column):
    counts = df[column].value_counts()
    single_offer = (counts == 1).sum()
    total = len(counts)
    return single_offer, (single_offer / total) * 100

single, percent = analyze_long_tail(df, 'empresa')
print(f"Empresas con una sola oferta: {single} ({percent:.2f}% del total)")

Empresas con una sola oferta: 39 (88.64% del total)


## 3. Tech Stack

Análisis de tecnologías, lenguajes y frameworks requeridos.

<div style="font-size: 0.8em; line-height: 1.8; color: #ffffffff; font-family: monospace; text-align: right; padding-right: 10px; margin-top: 10px;">
Triple JOIN: Unión de ofertas_tecnologias, tecnologias y ofertas para obtener el contexto completo.
</div>

In [11]:
# 3.0 Carga de Datos Tech
query_techs = """
SELECT ot.oferta_id, t.nombre as tech, o.origen_proceso
FROM ofertas_tecnologias ot
JOIN tecnologias t ON ot.tecnologia_id = t.id
JOIN ofertas o ON ot.oferta_id = o.id
"""
df_techs = pd.read_sql(query_techs, engine)
print(f"Carga Tech Exitosa: {len(df_techs)} registros de tecnologías.")

Carga Tech Exitosa: 227 registros de tecnologías.


<div style="font-size: 0.8em; line-height: 1.8; color: #ffffffff; font-family: monospace; text-align: right; padding-right: 10px; margin-top: 10px;">
Popularidad: Conteo directo de menciones por tecnología.
</div>

In [12]:
# 3.1 Popularidad (Top 15)
print("Tecnologías más demandadas:")
print(df_techs['tech'].value_counts().head(15))

Tecnologías más demandadas:
tech
Git           23
JavaScript    21
SQL           17
PHP           14
PostgreSQL    13
Python        12
MySQL         12
React         12
HTML          11
CSS           10
Node.js        8
Docker         8
MongoDB        8
Azure          7
Laravel        6
Name: count, dtype: int64


<div style="font-size: 0.8em; line-height: 1.8; color: #ffffffff; font-family: monospace; text-align: right; padding-right: 10px; margin-top: 10px;">
Counter(): Objeto para contar elementos de forma eficiente.<br>
combinations(): Genera pares únicos de tecnologías por oferta.
</div>

In [13]:
# 3.2 Combinaciones Frecuentes
from itertools import combinations
from collections import Counter

groups = df_techs.groupby('oferta_id')['tech'].apply(list)
combos = Counter()
for techs in groups:
    if len(techs) > 1:
        combos.update(combinations(sorted(techs), 2))

print("Pares de tecnologías que suelen ir juntos:")
for combo, count in combos.most_common(10):
    print(f"{combo}: {count}")

Pares de tecnologías que suelen ir juntos:
('Git', 'JavaScript'): 14
('JavaScript', 'PHP'): 12
('JavaScript', 'MySQL'): 11
('JavaScript', 'PostgreSQL'): 11
('Git', 'MySQL'): 10
('HTML', 'JavaScript'): 10
('Git', 'SQL'): 9
('CSS', 'Git'): 9
('CSS', 'HTML'): 9
('CSS', 'JavaScript'): 9


<div style="font-size: 0.8em; line-height: 1.8; color: #ffffffff; font-family: monospace; text-align: right; padding-right: 10px; margin-top: 10px;">
round(): Redondea el promedio a decimales legibles.
</div>

In [14]:
# 3.3 Promedio de Tecnologías
avg_techs = df_techs.groupby('oferta_id')['tech'].count().mean()
print(f"Cantidad promedio de tecnologías por oferta: {avg_techs:.2f}")

Cantidad promedio de tecnologías por oferta: 5.40


<div style="font-size: 0.8em; line-height: 1.8; color: #ffffffff; font-family: monospace; text-align: right; padding-right: 10px; margin-top: 10px;">
Rareza: Filtro de tecnologías con presencia marginal (<1%).
</div>

In [15]:
# 3.4 Tecnologías Raras
total_offers = df_techs['oferta_id'].nunique()
counts = df_techs['tech'].value_counts()
ratios = (counts / total_offers) * 100
print("Tecnologías presentes en menos del 1% de las ofertas:")
print(ratios[ratios < 1])

Tecnologías presentes en menos del 1% de las ofertas:
Series([], Name: count, dtype: float64)


<div style="font-size: 0.8em; line-height: 1.8; color: #ffffffff; font-family: monospace; text-align: right; padding-right: 10px; margin-top: 10px;">
lambda: Función anónima para extraer el Top 5 por cada origen.
</div>

In [16]:
# 3.5 Tendencia por Origen
trends = df_techs.groupby('origen_proceso')['tech'].apply(lambda x: x.value_counts().head(5))
print("Top 5 tecnologías por origen de proceso:")
print(trends)

Top 5 tecnologías por origen de proceso:
origen_proceso            
dds             Azure          5
                Git            3
                SQL            3
                AWS            3
                JavaScript     2
dds_full        JavaScript     8
                Git            7
                PHP            5
                HTML           5
                SQL            5
fullstack       Git           13
                JavaScript    11
                React         11
                SQL            9
                Python         8
Name: tech, dtype: int64


## 3. Tech Stack

Análisis de tecnologías, lenguajes y frameworks requeridos.

<div style="font-size: 0.8em; line-height: 1.8; color: #ffffffff; font-family: monospace; text-align: right; padding-right: 10px; margin-top: 10px;">
Triple JOIN: Cruce entre ofertas_tecnologias, tecnologias y ofertas.
</div>

In [17]:
# 3.0 Carga de Datos Tech
query_techs = """
SELECT ot.oferta_id, t.nombre as tech, o.origen_proceso
FROM ofertas_tecnologias ot
JOIN tecnologias t ON ot.tecnologia_id = t.id
JOIN ofertas o ON ot.oferta_id = o.id
"""
df_techs = pd.read_sql(query_techs, engine)
print(f"Carga Tech Exitosa: {len(df_techs)} registros.")

Carga Tech Exitosa: 227 registros.


<div style="font-size: 0.8em; line-height: 1.8; color: #ffffffff; font-family: monospace; text-align: right; padding-right: 10px; margin-top: 10px;">
Popularidad: Top 15 de tecnologías más demandadas.
</div>

In [18]:
# 3.1 Popularidad
print(df_techs['tech'].value_counts().head(15))

tech
Git           23
JavaScript    21
SQL           17
PHP           14
PostgreSQL    13
Python        12
MySQL         12
React         12
HTML          11
CSS           10
Node.js        8
Docker         8
MongoDB        8
Azure          7
Laravel        6
Name: count, dtype: int64


<div style="font-size: 0.8em; line-height: 1.8; color: #ffffffff; font-family: monospace; text-align: right; padding-right: 10px; margin-top: 10px;">
Counter + combinations: Identifica pares que co-ocurren por oferta.
</div>

In [19]:
# 3.2 Combinaciones
from itertools import combinations
from collections import Counter

groups = df_techs.groupby('oferta_id')['tech'].apply(list)
combos = Counter()
for techs in groups:
    if len(techs) > 1:
        combos.update(combinations(sorted(techs), 2))

print("Pares más frecuentes:")
print(combos.most_common(10))

Pares más frecuentes:
[(('Git', 'JavaScript'), 14), (('JavaScript', 'PHP'), 12), (('JavaScript', 'MySQL'), 11), (('JavaScript', 'PostgreSQL'), 11), (('Git', 'MySQL'), 10), (('HTML', 'JavaScript'), 10), (('Git', 'SQL'), 9), (('CSS', 'Git'), 9), (('CSS', 'HTML'), 9), (('CSS', 'JavaScript'), 9)]


<div style="font-size: 0.8em; line-height: 1.8; color: #ffffffff; font-family: monospace; text-align: right; padding-right: 10px; margin-top: 10px;">
Promedio: Media de tecnologías solicitadas por vacante.
</div>

In [20]:
# 3.3 Promedio
avg_techs = df_techs.groupby('oferta_id')['tech'].count().mean()
print(f"Promedio: {avg_techs:.2f} techs/oferta")

Promedio: 5.40 techs/oferta


<div style="font-size: 0.8em; line-height: 1.8; color: #ffffffff; font-family: monospace; text-align: right; padding-right: 10px; margin-top: 10px;">
Rareza: Tecnologías con presencia marginal (<1%).
</div>

In [21]:
# 3.4 Rareza
total_offers = df_techs['oferta_id'].nunique()
counts = df_techs['tech'].value_counts()
ratios = (counts / total_offers) * 100
print(ratios[ratios < 1])

Series([], Name: count, dtype: float64)


<div style="font-size: 0.8em; line-height: 1.8; color: #ffffffff; font-family: monospace; text-align: right; padding-right: 10px; margin-top: 10px;">
Tendencia: Distribución del stack según el origen del proceso.
</div>

In [22]:
# 3.5 Tendencia
print(df_techs.groupby('origen_proceso')['tech'].apply(lambda x: x.value_counts().head(5)))

origen_proceso            
dds             Azure          5
                Git            3
                SQL            3
                AWS            3
                JavaScript     2
dds_full        JavaScript     8
                Git            7
                PHP            5
                HTML           5
                SQL            5
fullstack       Git           13
                JavaScript    11
                React         11
                SQL            9
                Python         8
Name: tech, dtype: int64


<div style="font-size: 0.8em; line-height: 1.8; color: #ffffffff; font-family: monospace; text-align: right; padding-right: 10px; margin-top: 10px;">
Unicidad: Total de tecnologías identificadas en el ecosistema.
</div>

In [23]:
# 3.6 Unicidad
print(f"Total tecnologías únicas: {df_techs['tech'].nunique()}")

Total tecnologías únicas: 32


<div style="font-size: 0.8em; line-height: 1.8; color: #ffffffff; font-family: monospace; text-align: right; padding-right: 10px; margin-top: 10px;">
Densidad (Mín/Máx): Rango de carga técnica por vacante.
</div>

In [24]:
# 3.7 Densidad
counts = df_techs.groupby('oferta_id')['tech'].count()
print(f"Rango de tecnologías: {counts.min()} (mín) a {counts.max()} (máx)")

Rango de tecnologías: 1 (mín) a 15 (máx)


## 4. Experiencia en Años

Análisis de los años de trayectoria solicitados en las vacantes.

<div style="font-size: 0.8em; line-height: 1.8; color: #ffffffff; font-family: monospace; text-align: right; padding-right: 10px; margin-top: 10px;">
.mean() / .median(): Medidas de tendencia central para la experiencia.
</div>

In [25]:
# 4.1 Promedio y Mediana
avg_exp = df['experiencia_anios'].mean()
med_exp = df['experiencia_anios'].median()
print(f"Promedio: {avg_exp:.2f} años | Mediana: {med_exp} años")

Promedio: 3.57 años | Mediana: 3.0 años


<div style="font-size: 0.8em; line-height: 1.8; color: #ffffffff; font-family: monospace; text-align: right; padding-right: 10px; margin-top: 10px;">
pd.cut(): Segmenta valores en intervalos definidos (Junior, Middle, Senior).
</div>

In [26]:
# 4.2 Distribución por Niveles
bins = [0, 1.9, 4.9, 100]
labels = ['Junior (0-2)', 'Middle (2-5)', 'Senior (5+)']
print(pd.cut(df['experiencia_anios'], bins=bins, labels=labels).value_counts())

experiencia_anios
Middle (2-5)    18
Junior (0-2)    11
Senior (5+)      7
Name: count, dtype: int64


<div style="font-size: 0.8em; line-height: 1.8; color: #ffffffff; font-family: monospace; text-align: right; padding-right: 10px; margin-top: 10px;">
.corr(): Evalúa si a mayor experiencia se solicitan más tecnologías.
</div>

In [27]:
# 4.3 Correlación (Experiencia vs Cantidad Techs)
tech_counts = df_techs.groupby('oferta_id')['tech'].count()
df['num_techs'] = df['id'].map(tech_counts).fillna(0)
correlation = df['experiencia_anios'].corr(df['num_techs'])
print(f"Correlación de Pearson: {correlation:.4f}")

Correlación de Pearson: 0.0826


<div style="font-size: 0.8em; line-height: 1.8; color: #ffffffff; font-family: monospace; text-align: right; padding-right: 10px; margin-top: 10px;">
Extremos: Identifica el techo de experiencia y su stack tecnológico.
</div>

In [28]:
# 4.4 Extremos
max_exp = df['experiencia_anios'].max()
ids_max = df[df['experiencia_anios'] == max_exp]['id']
techs_max = df_techs[df_techs['oferta_id'].isin(ids_max)]['tech'].unique()
print(f"Experiencia Máxima: {max_exp} años")
print(f"Tecnologías en ofertas de {max_exp} años: {list(techs_max)}")

Experiencia Máxima: 31.0 años
Tecnologías en ofertas de 31.0 años: ['.NET', 'SQL', 'MySQL', 'Azure', 'Git']


<div style="font-size: 0.8em; line-height: 1.8; color: #ffffffff; font-family: monospace; text-align: right; padding-right: 10px; margin-top: 10px;">
(df[col] == 0).mean(): Proporción de ofertas para perfiles sin experiencia.
</div>

In [29]:
# 4.5 Entry Level Rate
entry_rate = (df['experiencia_anios'] == 0).mean() * 100
print(f"Porcentaje de ofertas Entry Level (0 años): {entry_rate:.2f}%")

Porcentaje de ofertas Entry Level (0 años): 0.00%


<div style="font-size: 0.8em; line-height: 1.8; color: #ffffffff; font-family: monospace; text-align: right; padding-right: 10px; margin-top: 10px;">
.std(): Desviación estándar para medir la variabilidad del mercado.
</div>

In [30]:
# 4.6 Volatilidad (Desviación Estándar)
std_exp = df['experiencia_anios'].std()
print(f"Desviación Estándar de la experiencia: {std_exp:.2f} años")

Desviación Estándar de la experiencia: 5.06 años


<div style="font-size: 0.8em; line-height: 1.8; color: #ffffffff; font-family: monospace; text-align: right; padding-right: 10px; margin-top: 10px;">
Moda: El valor de años más solicitado estadísticamente.
</div>

In [31]:
# 4.7 Moda
mode_exp = df['experiencia_anios'].mode()[0]
print(f"Moda de la experiencia: {mode_exp} años")

Moda de la experiencia: 1.0 años


## 5. Compatibilidad

Análisis del puntaje de ajuste de las ofertas al perfil del usuario.

<div style="font-size: 0.8em; line-height: 1.8; color: #ffffffff; font-family: monospace; text-align: right; padding-right: 10px; margin-top: 10px;">
JOIN: Cruce entre compatibilidades y ofertas para obtener el origen del proceso.
</div>

In [32]:
# 5.0 Carga de Datos de Compatibilidad
query_compat = """
SELECT c.score, o.origen_proceso
FROM compatibilidades c
JOIN ofertas o ON c.oferta_id = o.id
"""
df_compat = pd.read_sql(query_compat, engine)
print(f"Carga Exitosa: {len(df_compat)} registros de compatibilidad.")

Carga Exitosa: 116 registros de compatibilidad.


<div style="font-size: 0.8em; line-height: 1.8; color: #ffffffff; font-family: monospace; text-align: right; padding-right: 10px; margin-top: 10px;">
.mean() / .median(): Medidas de tendencia central.
</div>

In [33]:
# 5.1 Promedio y Mediana
avg_comp = df_compat['score'].mean()
med_comp = df_compat['score'].median()
print(f"Promedio: {avg_comp:.2f}% | Mediana: {med_comp:.2f}%")

Promedio: 0.37% | Mediana: 0.38%


<div style="font-size: 0.8em; line-height: 1.8; color: #ffffffff; font-family: monospace; text-align: right; padding-right: 10px; margin-top: 10px;">
Filtro Booleano: Cuenta cuántas ofertas superan el umbral establecido.
</div>

In [34]:
# 5.2 Top Ofertas (>70%)
top_count = (df_compat['score'] >= 70).sum()
print(f"Ofertas Top (>=70%): {top_count}")

Ofertas Top (>=70%): 0


<div style="font-size: 0.8em; line-height: 1.8; color: #ffffffff; font-family: monospace; text-align: right; padding-right: 10px; margin-top: 10px;">
.groupby().mean(): Compara promedios por categoría (Origen de Proceso).
</div>

In [35]:
# 5.3 Desempeño por Origen
perf = df_compat.groupby('origen_proceso')['score'].mean().sort_values(ascending=False)
print("Promedio de compatibilidad por origen:")
print(perf)

Promedio de compatibilidad por origen:
origen_proceso
dds          0.388355
dds_full     0.380089
fullstack    0.336955
Name: score, dtype: float64


<div style="font-size: 0.8em; line-height: 1.8; color: #ffffffff; font-family: monospace; text-align: right; padding-right: 10px; margin-top: 10px;">
.std(): Mide qué tan dispersos están los scores.
</div>

In [36]:
# 5.4 Dispersión (Desviación Estándar)
std_comp = df_compat['score'].std()
print(f"Desviación Estándar: {std_comp:.2f}%")

Desviación Estándar: 0.20%


<div style="font-size: 0.8em; line-height: 1.8; color: #ffffffff; font-family: monospace; text-align: right; padding-right: 10px; margin-top: 10px;">
Tasa de Descarte: Porcentaje de ofertas consideradas "basura" para el perfil.
</div>

In [37]:
# 5.5 Tasa de Descarte (<40%)
bottom_rate = (df_compat['score'] < 40).mean() * 100
print(f"Ofertas Descartables (<40%): {bottom_rate:.2f}%")

Ofertas Descartables (<40%): 100.00%


<div style="font-size: 0.8em; line-height: 1.8; color: #ffffffff; font-family: monospace; text-align: right; padding-right: 10px; margin-top: 10px;">
Rango Absoluto: Identifica los límites reales del sistema.
</div>

In [38]:
# 5.6 Rango Absoluto
min_comp = df_compat['score'].min()
max_comp = df_compat['score'].max()
print(f"El score de compatibilidad va desde {min_comp}% hasta {max_comp}%")

El score de compatibilidad va desde 0.0% hasta 0.65%


## 6. Requerimiento de Inglés

Análisis del impacto de dominar un segundo idioma en las ofertas laborales.

<div style="font-size: 0.8em; line-height: 1.8; color: #ffffffff; font-family: monospace; text-align: right; padding-right: 10px; margin-top: 10px;">
JOIN Maestro: Cruzamos ofertas, compatibilidades y preparamos las métricas base.
</div>

In [39]:
# 6.0 Carga de Datos Bilingües
query_ingles = """
SELECT 
    o.id, 
    o.requiere_ingles, 
    o.experiencia_anios, 
    o.origen_proceso,
    c.score as compatibilidad
FROM ofertas o
LEFT JOIN compatibilidades c ON o.id = c.oferta_id
"""
df_ingles = pd.read_sql(query_ingles, engine)

# Aprovechamos la carga de df_techs previa (o simulamos el conteo si ya lo tenemos)
tech_counts = df_techs.groupby('oferta_id')['tech'].count()
df_ingles['num_techs'] = df_ingles['id'].map(tech_counts).fillna(0)
print(f"Carga exitosa: {len(df_ingles)} registros para análisis bilingüe.")

Carga exitosa: 116 registros para análisis bilingüe.


<div style="font-size: 0.8em; line-height: 1.8; color: #ffffffff; font-family: monospace; text-align: right; padding-right: 10px; margin-top: 10px;">
(df[col] == True).mean(): Porcentaje directo de booleanos verdaderos.
</div>

In [40]:
# 6.1 Proporción General
prop_ingles = (df_ingles['requiere_ingles'] == True).mean() * 100
print(f"El {prop_ingles:.2f}% de las ofertas exigen inglés.")

El 8.62% de las ofertas exigen inglés.


<div style="font-size: 0.8em; line-height: 1.8; color: #ffffffff; font-family: monospace; text-align: right; padding-right: 10px; margin-top: 10px;">
.groupby().mean(): Compara métricas objetivo dividiendo el dataset por un booleano.
</div>

In [41]:
# 6.2 Impacto en Años de Experiencia
exp_vs_ingles = df_ingles.groupby('requiere_ingles')['experiencia_anios'].mean()
print("Años de experiencia requeridos (Con inglés vs Sin inglés):")
print(exp_vs_ingles)

Años de experiencia requeridos (Con inglés vs Sin inglés):
requiere_ingles
False    3.390625
True     5.000000
Name: experiencia_anios, dtype: float64


<div style="font-size: 0.8em; line-height: 1.8; color: #ffffffff; font-family: monospace; text-align: right; padding-right: 10px; margin-top: 10px;">
Filtro + value_counts(): Identifica qué tecnologías son el 'core' internacional.
</div>

In [42]:
# 6.3 Relación Tecnológica (Top 10 Techs Bilingües)
ids_bilingues = df_ingles[df_ingles['requiere_ingles'] == True]['id']
top_techs_ingles = df_techs[df_techs['oferta_id'].isin(ids_bilingues)]['tech'].value_counts().head(10)
print("Top 10 Tecnologías en ofertas que exigen inglés:")
print(top_techs_ingles)

Top 10 Tecnologías en ofertas que exigen inglés:
tech
Git           2
Azure         2
AWS           2
PostgreSQL    2
JavaScript    1
Python        1
Flutter       1
Node.js       1
React         1
Docker        1
Name: count, dtype: int64


<div style="font-size: 0.8em; line-height: 1.8; color: #ffffffff; font-family: monospace; text-align: right; padding-right: 10px; margin-top: 10px;">
Concentración: Promedio del valor booleano agrupado por una categoría.
</div>

In [43]:
# 6.4 Concentración por Origen
concentracion = df_ingles.groupby('origen_proceso')['requiere_ingles'].mean() * 100
print("Porcentaje de ofertas en inglés por cada origen:")
print(concentracion.sort_values(ascending=False))

Porcentaje de ofertas en inglés por cada origen:
origen_proceso
dds          15.0
fullstack    10.0
dds_full      0.0
Name: requiere_ingles, dtype: float64


<div style="font-size: 0.8em; line-height: 1.8; color: #ffffffff; font-family: monospace; text-align: right; padding-right: 10px; margin-top: 10px;">
Brecha Técnica: Evalúa si el inglés trae consigo mayor carga de herramientas.
</div>

In [44]:
# 6.5 Brecha de Carga Técnica
techs_vs_ingles = df_ingles.groupby('requiere_ingles')['num_techs'].mean()
print("Cantidad promedio de tecnologías (Con inglés vs Sin inglés):")
print(techs_vs_ingles)

Cantidad promedio de tecnologías (Con inglés vs Sin inglés):
requiere_ingles
False    3.962264
True     3.400000
Name: num_techs, dtype: float64


<div style="font-size: 0.8em; line-height: 1.8; color: #ffffffff; font-family: monospace; text-align: right; padding-right: 10px; margin-top: 10px;">
Match Profile: Descubre estadísticamente en qué mercado encajas mejor.
</div>

In [45]:
# 6.6 Match Profile (Impacto en Compatibilidad)
score_vs_ingles = df_ingles.groupby('requiere_ingles')['compatibilidad'].mean()
print("Score de compatibilidad promedio (Con inglés vs Sin inglés):")
print(score_vs_ingles)

Score de compatibilidad promedio (Con inglés vs Sin inglés):
requiere_ingles
False    0.402789
True     0.000000
Name: compatibilidad, dtype: float64


## 7. Fecha de Publicación Estimada

Análisis temporal para medir el ritmo y la cadencia del mercado de ofertas.

<div style="font-size: 0.8em; line-height: 1.8; color: #ffffffff; font-family: monospace; text-align: right; padding-right: 10px; margin-top: 10px;">
Carga de fechas: Es vital asegurar el tipo datetime de Pandas al leer de SQL.
</div>

In [46]:
# 7.0 Carga de Fechas
query_fechas = "SELECT id, fecha_publicacion_estimada FROM ofertas"
df_fechas = pd.read_sql(query_fechas, engine)
df_fechas['fecha_publicacion_estimada'] = pd.to_datetime(df_fechas['fecha_publicacion_estimada'])
print(f"Carga exitosa: {len(df_fechas)} registros temporales.")

Carga exitosa: 58 registros temporales.


<div style="font-size: 0.8em; line-height: 1.8; color: #ffffffff; font-family: monospace; text-align: right; padding-right: 10px; margin-top: 10px;">
dt.day_name(): Extrae el nombre del día de la semana.
</div>

In [47]:
# 7.1 Volumen Temporal (Días de la semana)
vol_dias = df_fechas['fecha_publicacion_estimada'].dt.day_name().value_counts()
print("Volumen de ofertas por día de la semana:")
print(vol_dias)

Volumen de ofertas por día de la semana:
fecha_publicacion_estimada
Monday       21
Tuesday      20
Sunday        7
Saturday      3
Friday        3
Thursday      3
Wednesday     1
Name: count, dtype: int64


<div style="font-size: 0.8em; line-height: 1.8; color: #ffffffff; font-family: monospace; text-align: right; padding-right: 10px; margin-top: 10px;">
pd.Timestamp.now() - pd.Timedelta(): Filtra registros en ventanas de tiempo cortas.
</div>

In [48]:
# 7.2 Recencia (24 y 48 horas)
now = pd.Timestamp.now()
ultimas_24 = (df_fechas['fecha_publicacion_estimada'] >= (now - pd.Timedelta(hours=24))).sum()
ultimas_48 = (df_fechas['fecha_publicacion_estimada'] >= (now - pd.Timedelta(hours=48))).sum()
print(f"Ofertas publicadas en las últimas 24h: {ultimas_24}")
print(f"Ofertas publicadas en las últimas 48h: {ultimas_48}")

Ofertas publicadas en las últimas 24h: 0
Ofertas publicadas en las últimas 48h: 21


<div style="font-size: 0.8em; line-height: 1.8; color: #ffffffff; font-family: monospace; text-align: right; padding-right: 10px; margin-top: 10px;">
dt.hour: Extrae la hora en formato 24h para identificar franjas de actividad.
</div>

In [49]:
# 7.3 Frecuencia (Franjas Horarias)
frec_horas = df_fechas['fecha_publicacion_estimada'].dt.hour.value_counts().sort_index()
print("Actividad por hora del día (0-23):")
print(frec_horas)

Actividad por hora del día (0-23):
fecha_publicacion_estimada
5      2
6      1
7      1
9     21
10     1
11     1
12     1
14     1
16     2
17    25
18     1
23     1
Name: count, dtype: int64


<div style="font-size: 0.8em; line-height: 1.8; color: #ffffffff; font-family: monospace; text-align: right; padding-right: 10px; margin-top: 10px;">
(now - min()).days: Diferencia en días entre hoy y el registro más viejo.
</div>

In [50]:
# 7.4 Antigüedad Máxima (Frescura)
oldest = df_fechas['fecha_publicacion_estimada'].min()
max_age = (now - oldest).days if pd.notnull(oldest) else 0
print(f"El registro más antiguo tiene {max_age} días en el sistema.")

El registro más antiguo tiene 31 días en el sistema.


<div style="font-size: 0.8em; line-height: 1.8; color: #ffffffff; font-family: monospace; text-align: right; padding-right: 10px; margin-top: 10px;">
.mode()[0]: Extrae el día matemático de máxima actividad.
</div>

In [51]:
# 7.5 Día Pico Absoluto
peak_day = df_fechas['fecha_publicacion_estimada'].dt.day_name().mode()[0]
print(f"Día Pico Absoluto ('Golden Day'): {peak_day}")

Día Pico Absoluto ('Golden Day'): Monday


<div style="font-size: 0.8em; line-height: 1.8; color: #ffffffff; font-family: monospace; text-align: right; padding-right: 10px; margin-top: 10px;">
dt.dayofweek.isin([5, 6]): Filtra sábado (5) y domingo (6).
</div>

In [52]:
# 7.6 Fuga de Fin de Semana (Weekend Drop-off)
weekend_rate = df_fechas['fecha_publicacion_estimada'].dt.dayofweek.isin([5, 6]).mean() * 100
print(f"Tasa de publicaciones en fin de semana: {weekend_rate:.2f}%")

Tasa de publicaciones en fin de semana: 17.24%


## 8. Título de la Oferta

Análisis de NLP y normalización para los nombres de los cargos.

<div style="font-size: 0.8em; line-height: 1.8; color: #ffffffff; font-family: monospace; text-align: right; padding-right: 10px; margin-top: 10px;">
<strong>NOTA:</strong> Este campo requiere optimizaciones profundas (NLP, Stop Words, Normalización avanzada). Queda programado para la <strong>v.2</strong> del proyecto. La estructura se mantiene como placeholder.
</div>

## 9. ID de Oferta

Auditoría técnica de integridad sobre la llave primaria del dataset.

<div style="font-size: 0.8em; line-height: 1.8; color: #ffffffff; font-family: monospace; text-align: right; padding-right: 10px; margin-top: 10px;">
La integridad de IDs valida si el scraping trajo ofertas repetidas.
</div>

In [53]:
# 9.0 Carga de IDs
query_id = "SELECT id FROM ofertas"
df_id = pd.read_sql(query_id, engine)
print(f"Carga exitosa para auditoría de IDs.")

Carga exitosa para auditoría de IDs.


<div style="font-size: 0.8em; line-height: 1.8; color: #ffffffff; font-family: monospace; text-align: right; padding-right: 10px; margin-top: 10px;">
len() vs .nunique(): Base del cálculo de integridad.
</div>

In [54]:
# 9.1 Integridad Completa (Volumen, Unicidad, Duplicidad y Tasa)
total_ids = len(df_id)
unicos_ids = df_id['id'].nunique()
duplicados_ids = total_ids - unicos_ids
tasa_duplicidad = (duplicados_ids / total_ids) * 100 if total_ids > 0 else 0

print(f"[Volumen] Total Registros: {total_ids}")
print(f"[Unicidad] Registros Únicos: {unicos_ids}")
print(f"[Duplicidad] Detectados: {duplicados_ids} (Tasa: {tasa_duplicidad:.2f}%)")

[Volumen] Total Registros: 58
[Unicidad] Registros Únicos: 58
[Duplicidad] Detectados: 0 (Tasa: 0.00%)
